# SCD Type 6 — Hybrid (Type 1 + Type 2 + Type 3)

**SCD Type 6** is named after **1 + 2 + 3 = 6** — it combines all three approaches into one table.

It keeps the **full row history** of Type 2, but also adds:
- A `current_name` column (Type 1 style) that is **updated on every historical row** to always reflect the latest value — no filter needed to get the current name.
- A `previous_name` column (Type 3 style) showing what the value was before the most recent change.

### The problem Type 6 solves
In pure Type 2 you must always filter `WHERE is_current = TRUE` to get the current name.  
If analysts join a fact table (which stores `surrogate_key`) to the dimension, they need extra logic to pick the right version.  
With Type 6, joining on `surrogate_key` **always gives you the current name** via `current_name`, while `name` still holds the historically-accurate value for that version.

### Table structure
| Column | Purpose |
|---|---|
| `surrogate_key` | Auto-generated unique row identifier |
| `id` | Natural/business key |
| `name` | The value **at the time this version was created** (Type 2 historical value) |
| `current_name` | **Always** the latest value — updated across all rows on change (Type 1) |
| `previous_name` | The value before the most recent change (Type 3) |
| `effective_start` | When this version became active |
| `effective_end` | When this version was superseded (`NULL` = still current) |
| `is_current` | `TRUE` for the active row |

### Techniques covered
1. Create the Type 6 Delta table
2. Initial load from source CSV
3. SCD Type 6 MERGE — expire old rows, insert new versions, update `current_name` on all rows
4. Query: current snapshot (simple — no filter on `is_current` needed for `current_name`)
5. Query: full history with accurate historical `name` vs always-current `current_name`
6. Query: analytical comparison across historical rows
7. Delta time travel
8. Delta history

## Setup

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

DATA_ROOT = "/Volumes/population_metrics/landing/datasets"

from pyspark.sql import functions as F, Row
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from delta.tables import DeltaTable

spark.sql("SELECT current_catalog(), current_schema()").show(truncate=False)

## 1. Read the source CSV

In [ ]:
CSV_PATH = f"{DATA_ROOT}/countries_dataset/countries_dataset/csv_data/country_regions/country_regions.csv"

source_schema = StructType([
    StructField("id",   IntegerType(), False),
    StructField("name", StringType(),  True)
])

source_df = spark.read.format("csv").schema(source_schema).option("header", True).load(CSV_PATH)
source_df.show()

## 2. Create the SCD Type 6 Delta table

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS population_metrics.default.scd6_country_regions (
        surrogate_key   BIGINT GENERATED ALWAYS AS IDENTITY,
        id              INT       NOT NULL,
        name            STRING,
        current_name    STRING,
        previous_name   STRING,
        effective_start TIMESTAMP NOT NULL,
        effective_end   TIMESTAMP,
        is_current      BOOLEAN   NOT NULL
    )
    USING DELTA
    COMMENT 'SCD Type 6 (Hybrid 1+2+3) dimension table for world regions'
""")
print("Table ready.")

## 3. Initial load

On first load `name`, `current_name`, and `previous_name` are all set consistently:
- `name` = original value (historical)
- `current_name` = same (no change has occurred yet)
- `previous_name` = NULL

In [ ]:
row_count = spark.table("population_metrics.default.scd6_country_regions").count()

if row_count == 0:
    initial_df = source_df.select(
        F.col("id"),
        F.col("name"),
        F.col("name").alias("current_name"),
        F.lit(None).cast("string").alias("previous_name"),
        F.current_timestamp().alias("effective_start"),
        F.lit(None).cast("timestamp").alias("effective_end"),
        F.lit(True).alias("is_current")
    )
    initial_df.write.format("delta").mode("append").saveAsTable("population_metrics.default.scd6_country_regions")
    print(f"Seeded {initial_df.count()} rows.")
else:
    print(f"Table already has {row_count} rows — skipping initial load.")

spark.table("population_metrics.default.scd6_country_regions").orderBy("id").show(truncate=False)

## 4. SCD Type 6 MERGE — three-step process

When the source delivers an update, three things must happen atomically:

1. **Expire** the current row for any changed `id` (set `effective_end`, `is_current = FALSE`).
2. **Insert** a new current row for the changed `id` with the updated name.
3. **Back-fill** `current_name` on **all historical rows** for that `id` so they all reflect the latest value.

Simulated change: America → *North & South America*, Antarctica (id=60) added.

In [ ]:
updated_rows = [
    Row(id=10, name="North & South America"),  # changed
    Row(id=20, name="Europe"),
    Row(id=30, name="Asia"),
    Row(id=40, name="Oceania"),
    Row(id=50, name="Africa"),
    Row(id=60, name="Antarctica"),              # new
]
updated_df = spark.createDataFrame(updated_rows, schema=source_schema)

scd6 = DeltaTable.forName(spark, "population_metrics.default.scd6_country_regions")

# ── Step 1: Expire changed rows, insert brand-new ids ───────────────────────
scd6.alias("target").merge(
    updated_df.alias("source"),
    "target.id = source.id AND target.is_current = TRUE"
).whenMatchedUpdate(
    condition="target.name <> source.name",
    set={
        "effective_end": "current_timestamp()",
        "is_current":    "FALSE"
    }
).whenNotMatchedInsert(
    values={
        "id":              "source.id",
        "name":            "source.name",
        "current_name":    "source.name",
        "previous_name":   "NULL",
        "effective_start": "current_timestamp()",
        "effective_end":   "NULL",
        "is_current":      "TRUE"
    }
).execute()

print("Step 1 complete — old rows expired, new ids inserted.")

In [ ]:
# ── Step 2: Insert new current rows for ids that were just expired ───────────
# Find ids that are now expired with no current version
expired_ids = (
    spark.table("population_metrics.default.scd6_country_regions")
    .filter("is_current = FALSE")
    .select("id")
    .join(
        spark.table("population_metrics.default.scd6_country_regions")
        .filter("is_current = TRUE").select("id"),
        on="id", how="left_anti"
    )
    .distinct()
)

# Get the previous (just-expired) name for these ids to populate previous_name
previous_names = (
    spark.table("population_metrics.default.scd6_country_regions")
    .filter("is_current = FALSE")
    .join(expired_ids, on="id", how="inner")
    .select("id", F.col("name").alias("prev_name"))
)

new_versions_df = (
    updated_df.alias("src")
    .join(expired_ids, on="id", how="inner")
    .join(previous_names, on="id", how="left")
    .select(
        F.col("src.id"),
        F.col("src.name"),
        F.col("src.name").alias("current_name"),
        F.col("prev_name").alias("previous_name"),
        F.current_timestamp().alias("effective_start"),
        F.lit(None).cast("timestamp").alias("effective_end"),
        F.lit(True).alias("is_current")
    )
)

if new_versions_df.count() > 0:
    new_versions_df.write.format("delta").mode("append").saveAsTable("population_metrics.default.scd6_country_regions")
    print(f"Step 2 complete — inserted {new_versions_df.count()} new current version(s).")
else:
    print("Step 2 — no changed rows to re-insert.")

In [ ]:
# ── Step 3 (Type 1 back-fill): Update current_name on ALL historical rows ────
# Build a map of id → latest name from the source
latest_names = updated_df.select("id", F.col("name").alias("latest_name"))

scd6.alias("target").merge(
    latest_names.alias("src"),
    "target.id = src.id AND target.current_name <> src.latest_name"
).whenMatchedUpdate(
    set={"current_name": "src.latest_name"}
).execute()

print("Step 3 complete — current_name back-filled on all historical rows.")
print()
spark.table("population_metrics.default.scd6_country_regions").orderBy("id", "effective_start").show(truncate=False)

## 5. Query the current snapshot — no `is_current` filter needed for current name

Because `current_name` is kept up-to-date on every row, you can get the current name of any region regardless of which surrogate key a fact row was stamped with.

In [ ]:
print("=== Current dimension (is_current = TRUE) ===")
spark.sql("""
    SELECT surrogate_key, id, name, current_name, previous_name, effective_start
    FROM population_metrics.default.scd6_country_regions
    WHERE is_current = TRUE
    ORDER BY id
""").show(truncate=False)

## 6. Full history — historical `name` vs always-current `current_name`

This query shows the key value of Type 6:  
- `name` = what the region was called **at the time of that version** (historically accurate)
- `current_name` = what the region is called **today** (on every row, even old ones)

In [ ]:
spark.sql("""
    SELECT
        surrogate_key,
        id,
        name            AS historical_name,
        current_name    AS todays_name,
        previous_name,
        effective_start,
        effective_end,
        is_current
    FROM population_metrics.default.scd6_country_regions
    ORDER BY id, effective_start
""").show(truncate=False)

## 7. Analytical query — fact table join scenario

Demonstrate why `current_name` is useful: imagine a fact table that recorded sales using a historical surrogate key (e.g., surrogate_key = 1 which was stamped when the region was still called "America").  
Joining on `surrogate_key` still gives us the *current* name without any extra logic.

In [ ]:
# Simulate a fact table with old surrogate keys
fact_data = [
    Row(sale_id=101, region_surrogate_key=1, amount=5000),   # was stamped when sk=1 = "America"
    Row(sale_id=102, region_surrogate_key=2, amount=3200),   # Europe
    Row(sale_id=103, region_surrogate_key=3, amount=7800),   # Asia
]
fact_schema = StructType([
    StructField("sale_id",              IntegerType(), False),
    StructField("region_surrogate_key", IntegerType(), False),
    StructField("amount",              IntegerType(), True)
])
fact_df = spark.createDataFrame(fact_data, schema=fact_schema)

dim_df = spark.table("population_metrics.default.scd6_country_regions")

result = fact_df.alias("f").join(
    dim_df.alias("d"),
    F.col("f.region_surrogate_key") == F.col("d.surrogate_key"),
    how="left"
).select(
    F.col("f.sale_id"),
    F.col("f.amount"),
    F.col("d.historical_name").alias("region_at_sale_time") if "historical_name" in dim_df.columns
        else F.col("d.name").alias("region_at_sale_time"),
    F.col("d.current_name").alias("region_current_name")
)

result.show(truncate=False)
print("region_at_sale_time = name when the sale happened (historical)")
print("region_current_name = name today (Type 1 back-fill via Type 6)")

## 8. Delta Lake time travel — see the table before back-fill

In [ ]:
# Version 1 = right after initial seed (no changes yet)
spark.sql("""
    SELECT surrogate_key, id, name, current_name, is_current
    FROM population_metrics.default.scd6_country_regions VERSION AS OF 1
    ORDER BY id
""").show(truncate=False)

## 9. Delta transaction log

In [ ]:
spark.sql("""
    DESCRIBE HISTORY population_metrics.default.scd6_country_regions
""").select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

## 10. Summary — all four SCD types compared

| | Type 1 | Type 2 | Type 3 | Type 6 |
|---|---|---|---|---|
| History kept | None | Full | One step | Full |
| Rows per key | 1 | Grows | 1 | Grows |
| Current value query | Simple | Needs `is_current` filter | Simple | Simple (`current_name`) |
| Historical value | Lost | `name` column | `previous_name` only | `name` column |
| Complexity | Low | Medium | Low | High |
| Best for | Corrections | Audits / compliance | Before-after reports | BI/analytics where easy joins matter |

## 11. (Optional) Cleanup

In [ ]:
# Uncomment to reset
# spark.sql("DROP TABLE IF EXISTS population_metrics.default.scd6_country_regions")
# print("Table dropped.")